# HDFC Bank: Fraud Risk Analysis - Part 5

**Objective:** Real-Time Gateway Model (Industry Standard)

We are discarding all complex `V` and `C` features because they cannot be computed in real-time without massive infrastructure (Feature Stores). Instead, we will train a highly interpretable, lightning-fast XGBoost model using **ONLY** features that are immediately available in the JSON payload of a payment request (<10ms).

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.engineering import build_feature_pipeline
from fraudguard.models.training import train_xgboost
from fraudguard.models.evaluation import evaluate_model

# MLflow Setup
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("FraudGuard_Gateway_Model")

<Experiment: artifact_location='mlflow-artifacts:/309966079389239364', creation_time=1788198951909, effective_trace_archival_retention=None, experiment_id='309966079389239364', last_update_time=1788198951909, lifecycle_stage='active', name='FraudGuard_Gateway_Model', tags={}, trace_location=None, workspace='default'>

## 1. The Gateway Feature Whitelist
These features are 100% human readable and instantly available at the payment gateway.

In [2]:
gateway_numeric_features = [
    'TransactionAmt', 
    'dist1', 'dist2'
]

# Note: The CustomFeatureEngineer (inside build_feature_pipeline) will automatically add 'Days', 'Hour'

gateway_categorical_features = [
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo'
]

print(f"Total whitelisted features: {len(gateway_numeric_features) + len(gateway_categorical_features)}")

Total whitelisted features: 25


## 2. Data Preparation

In [3]:
# 1. Load Data
data_dir = Path.cwd().parent / "data" / "raw"
df = load_bank_data(data_dir)

# 2. Split Temporally
df_train, df_test = split_temporal(df, test_ratio=0.2)
X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud'].values
X_test = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud'].values

# Ensure all whitelisted features exist in the dataframe before proceeding
# (Drop any missing ones gracefully to avoid KeyErrors if they don't exist in our raw data)
gateway_numeric_features = [f for f in gateway_numeric_features if f in X_train.columns]
gateway_categorical_features = [f for f in gateway_categorical_features if f in X_train.columns]

# 3. Build and Apply Pipeline
pipeline = build_feature_pipeline(gateway_numeric_features, gateway_categorical_features)
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)
print(f"Processed Matrix Shape: {X_train_processed.shape}")

Processed Matrix Shape: (472432, 25)


## 3. Train & Log XGBoost Champion Model

In [5]:
print("Training Final Gateway Model...")

with mlflow.start_run(run_name="XGBoost_Gateway_V1"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("feature_strategy", "real_time_gateway")
    
    xgb_model = train_xgboost(X_train_processed, y_train, X_val=X_test_processed, y_val=y_test)
    xgb_probs = xgb_model.predict_proba(X_test_processed)[:, 1]
    xgb_metrics = evaluate_model(y_test, xgb_probs, threshold=0.5)
    mlflow.log_metrics(xgb_metrics)
    
    from sklearn.pipeline import Pipeline
    full_deployable_model = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('classifier', xgb_model)
    ])
    
    # Save the pipeline object that we will deploy in Milestone 5
    mlflow.sklearn.log_model(full_deployable_model, "gateway_model")
    print("\nFinal Gateway Model Logged! Check MLflow UI.")

Training Final Gateway Model...
[0]	validation_0-aucpr:0.17921
[50]	validation_0-aucpr:0.21362
[100]	validation_0-aucpr:0.22921
[150]	validation_0-aucpr:0.24013
[200]	validation_0-aucpr:0.24530
[250]	validation_0-aucpr:0.24813
[300]	validation_0-aucpr:0.25115
[350]	validation_0-aucpr:0.25534
[400]	validation_0-aucpr:0.25981
[450]	validation_0-aucpr:0.26766
[500]	validation_0-aucpr:0.26732
[507]	validation_0-aucpr:0.26815


2026/08/31 23:54:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Gateway_V1 at: http://127.0.0.1:5000/#/experiments/309966079389239364/runs/6757c3f862c54080b233ebb7aebaebaa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/309966079389239364


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'